# Pydantic AI: Agents, Tools, Orchestration, and Structured Outputs

This notebook introduces Pydantic AI with OpenAI models. It uses dependencies from the repository-level `requirements.in` file and loads credentials with `python-dotenv`.

## Setup

Required environment variables:

- `OPENAI_API_KEY`
- optional `OPENAI_MODEL`, defaulting to `openai:gpt-5-nano`

The notebook does not install packages at runtime. Install the repository requirements before running it.

Because Jupyter already runs an asyncio event loop, this notebook uses top-level `await agent.run(...)` instead of `agent.run_sync(...)`.

In [1]:
import os
from dataclasses import dataclass
from datetime import date
from typing import Literal

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from pydantic_ai import Agent, RunContext, UsageLimits

In [2]:
load_dotenv()

True

In [3]:
def openai_model_name() -> str:
    model_name = os.getenv("OPENAI_MODEL", "openai:gpt-5-nano")
    if not model_name.startswith("openai:"):
        model_name = f"openai:{model_name}"
    return model_name

OPENAI_MODEL = openai_model_name()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    raise RuntimeError("Set OPENAI_API_KEY in your environment or .env file before running this notebook.")

print(OPENAI_MODEL)

openai:gpt-5-nano


## 1. Agent Generation

An `Agent` combines the model, instructions, optional dependency injection, optional tools, and optional structured output. Dynamic system prompts can read typed runtime dependencies through `RunContext`.

In [6]:
@dataclass
class LearnerProfile:
    name: str
    level: str
    goal: str


coach_agent = Agent(
    OPENAI_MODEL,
    deps_type=LearnerProfile,
    instructions="You are a concise AI learning coach. Give practical, beginner-safe guidance.",
)


@coach_agent.system_prompt
def add_learner_context(ctx: RunContext[LearnerProfile]) -> str:
    return (
        f"The learner is {ctx.deps.name}. "
        f"Level: {ctx.deps.level}. Goal: {ctx.deps.goal}. "
        f"Today's date is {date.today()}."
    )


profile = LearnerProfile(
    name="Mira",
    level="beginner",
    goal="understand when to use agent frameworks",
)

coach_result = await coach_agent.run(
    "Create a three-step study path for this week.",
    deps=profile,
)

print(coach_result.output)


Here’s a simple, beginner-friendly 3-step study path for this week to understand when to use agent frameworks.

Step 1 (Days 1–2): Learn the basics
- Goal: Know what an agent framework is and the core ideas behind it.
- What to do:
  - Read a short intro: what an agent, tools, memory, plan, and execution loop are.
  - Watch a quick 5–7 minute explainer video or a beginner article on agent frameworks (e.g., basics of how an agent can plan tasks and call tools).
  - Create a tiny glossary: write 5 key terms (agent, tool, memory, plan, execution loop) and your own simple definition for each.
- Quick outcome: You can explain in one sentence what an agent framework does and list its main parts.

Step 2 (Days 3–4): Explore use cases and do a mental demo
- Goal: Learn when it’s helpful to use an agent framework and when it’s not.
- What to do:
  - Pick 3 real-world tasks (for example: “answer a question using live data,” “combine info from two sources,” “perform a multi-step task over time”).

## 2. Tools

Use `@agent.tool_plain` for tools that do not need runtime context. Use `@agent.tool` when the function needs `RunContext`, injected dependencies, or shared application state.

In [7]:
@dataclass
class CourseCatalog:
    topics: dict[str, str]


tool_agent = Agent(
    OPENAI_MODEL,
    deps_type=CourseCatalog,
    instructions=(
        "You recommend course topics. Use tools when a catalog lookup or workload estimate "
        "would make the answer more reliable."
    ),
)


@tool_agent.tool_plain
def estimate_minutes(module_count: int, minutes_per_module: int) -> int:
    """Estimate total study time in minutes."""
    return module_count * minutes_per_module


@tool_agent.tool
def lookup_topic(ctx: RunContext[CourseCatalog], topic: str) -> str:
    """Look up a topic description from the course catalog."""
    return ctx.deps.topics.get(topic.lower(), "No catalog entry found for that topic.")


catalog = CourseCatalog(
    topics={
        "agents": "Agent systems combine model calls, tools, memory, and control flow.",
        "structured outputs": "Structured outputs return validated data instead of free-form prose.",
    }
)

tool_result = await tool_agent.run(
    "Explain agents and estimate the time for 4 modules at 25 minutes each.",
    deps=catalog,
)

print(tool_result.output)


Here’s a concise explanation of agents and the requested time estimate.

What is an agent?
- An agent is an autonomous or semi-autonomous system that acts on behalf of a user or itself to achieve goals. In practice, agent systems combine model calls, tools, memory, and control flow to perceive, decide, and act.
- Core components:
  - Goals or objectives: what the agent is trying to accomplish.
  - Observations/Environment: what the agent senses.
  - Decision-making/planner: how the agent decides what to do next.
  - Actions/Actuators: operations the agent can perform or commands it can issue.
  - Tools/Plugins: APIs or functions the agent can call to get information or perform tasks (e.g., web search, calculators, databases).
  - Memory: state or history kept to inform future decisions (short-term and/or long-term).
  - Control flow/orchestrator: manages sequencing, parallelism, and coordination between components.

How agents typically work
- Perceive input from the environment or use

## 3. Structured Outputs

`output_type` asks Pydantic AI to return data that validates against a Python type. Pydantic models are the most common option for application data.

In [9]:
class StudyPlan(BaseModel):
    topic: str = Field(description="The topic the learner should study.")
    objectives: list[str] = Field(description="Concrete learning objectives.")
    exercises: list[str] = Field(description="Practice tasks.")
    estimated_minutes: int = Field(ge=1, description="Estimated effort in minutes.")
    confidence: float = Field(ge=0, le=1, description="Confidence in the plan.")


planner_agent = Agent(
    OPENAI_MODEL,
    output_type=StudyPlan,
    instructions="Create compact, realistic study plans. Return only the requested structure.",
)

plan_result = await planner_agent.run(
    "Create a study plan for learning Pydantic AI tools and structured outputs."
)

study_plan = plan_result.output
print(study_plan.model_dump())

{'topic': 'Pydantic for AI tools and structured outputs', 'objectives': ['Master Pydantic v2 basics: BaseModel, Field, types, and validators', 'Define and validate structured schemas for AI outputs with strict validation', 'Leverage typing and Annotated for advanced field constraints and metadata', 'Configure models with model_config (aliases, extra behavior, validation modes)', 'Generate and validate JSON Schema from Pydantic models for data contracts', 'Parse and validate AI tool outputs (JSON and semi-structured) into Pydantic models', 'Design nested and reusable schemas for complex AI tasks and pipelines', 'Implement robust error handling, debugging, and logging for validation failures'], 'exercises': ['Create a simple Pydantic model for an AI response (id: str, text: str, confidence: float in [0,1], tags: List[str] | None)', 'Validate a sample AI output JSON against the model and handle validation errors', 'Add field aliases and validation constraints (min_length, max_length, boun

## 4. Agent Delegation

A coordinator agent can call another Pydantic AI agent through a tool. This keeps specialist behavior isolated while still giving the coordinator a simple action surface.

In [11]:
class SpecialistAnswer(BaseModel):
    answer: str
    next_step: str


policy_agent = Agent(
    OPENAI_MODEL,
    deps_type=dict[str, str],
    output_type=SpecialistAnswer,
    instructions="Answer from the supplied policy snippets. Be concise.",
)


@policy_agent.tool
def policy_lookup(ctx: RunContext[dict[str, str]], topic: str) -> str:
    """Look up a policy snippet by topic."""
    return ctx.deps.get(topic.lower(), "No policy snippet found.")


coordinator_agent = Agent(
    OPENAI_MODEL,
    instructions=(
        "You coordinate specialist help. Use ask_policy_specialist for policy questions, "
        "then summarize the answer for the user."
    ),
)


@coordinator_agent.tool_plain
async def ask_policy_specialist(question: str) -> str:
    """Ask the policy specialist agent for a grounded answer."""
    result = await policy_agent.run(
        question,
        deps={
            "refunds": "Refund requests must include an order ID and be filed within 30 days.",
            "security": "Security incidents must be escalated immediately to the security team.",
        },
        usage_limits=UsageLimits(request_limit=5),
    )
    return result.output.answer


delegation_result = await coordinator_agent.run(
    "A customer asks about the refund policy. Get specialist help and answer."
)

print(delegation_result.output)


I checked with our policy specialist. They can’t locate an exact official refund policy snippet in our system right now. To give you precise details (eligibility, timeframes, restocking fees or exclusions, refunds to the original payment method, and any regional differences), we need:

- Your region (e.g., US, EU, APAC)
- Your order number (optional but helpful)

If you’d prefer, you can also contact Support for the official policy directly. In the meantime, I can guide you through a generic refund process while we retrieve the exact policy.

What you can share with the customer right now (safe, generic template you can adapt):
- Our refund terms vary by region, so to provide exact details we’ll need your region and order number.
- A typical refund process (subject to official policy): 
  - Eligibility: item must meet the policy-allowed return criteria (e.g., within the return window, unused/ unopened where applicable; some items may be non-returnable).
  - Initiating a return: custome

## 5. Programmatic Handoff

Programmatic handoff keeps routing in application code. A router agent returns a typed decision, then the application calls the selected specialist.

In [12]:
class RouteDecision(BaseModel):
    route: Literal["policy", "learning"]
    reason: str


router_agent = Agent(
    OPENAI_MODEL,
    output_type=RouteDecision,
    instructions="Route the request to either policy or learning.",
)

learning_agent = Agent(
    OPENAI_MODEL,
    instructions="Explain technical learning topics clearly and briefly.",
)

user_question = "How should I learn structured outputs?"
route = (await router_agent.run(user_question)).output

if route.route == "policy":
    final_result = await policy_agent.run(user_question, deps={})
else:
    final_result = await learning_agent.run(user_question)

{
    "route": route.model_dump(),
    "final_answer": final_result.output,
}


{'route': {'route': 'learning',
  'reason': 'User asked for guidance on learning structured outputs; this is a learning-focused query.'},
 'final_answer': 'Structured outputs are outputs that have internal dependencies (sequences, trees, graphs, segmentations, etc.) rather than independent labels. Learning to predict them involves modeling those dependencies, doing efficient inference (decoding), and learning the parameters from data.\n\nWhat to learn (core ideas)\n- Output structure: how the parts of the output relate (e.g., sequences with Viterbi decoding, trees with dynamic programming, graphs with loopy inference).\n- Scoring and likelihood: how to assign scores or probabilities to complete outputs.\n- Inference (decoding): finding the best output under the model (Viterbi, beam search, graph-cut, etc.).\n- Learning: estimating model parameters from labeled data (maximum likelihood, CRF-style log-likelihood, structured SVM/hinge losses).\n- Evaluation: task-appropriate metrics (F1 f

## Recap

This notebook covered Pydantic AI agent creation, dependency injection, tools, structured outputs, agent delegation, programmatic handoff, and usage limits with OpenAI models loaded from environment variables.